In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
import tensorflow as tf
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
import os, sys, gc, time, warnings, pickle, psutil, random
from tqdm import tqdm
from math import ceil
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import math
from statsmodels.tsa.seasonal import STL
from scipy import stats
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))
print("GPU Details:", tf.config.experimental.list_physical_devices('GPU'))

Num GPUs Available: 1
GPU Details: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
cal = pd.read_parquet('done_cal.parquet')
# Đọc dữ liệu
grid_df = pd.read_parquet('../data/train_val.parquet')
grid_df = grid_df.drop(['date'], axis=1)
grid_df = grid_df.dropna(subset=['demand'])
print(grid_df.columns)
duplicate_id_list = grid_df[grid_df.duplicated('id', keep=False)]['id'].unique()
print(len(duplicate_id_list))

           date  wm_yr_wk  wday  month  year     d  event_name_1  \
0    2011-01-29     11101     1      1  2011     1             1   
1    2011-01-30     11101     2      1  2011     2             1   
2    2011-01-31     11101     3      1  2011     3             1   
3    2011-02-01     11101     4      2  2011     4             1   
4    2011-02-02     11101     5      2  2011     5             1   
...         ...       ...   ...    ...   ...   ...           ...   
1964 2016-06-15     11620     5      6  2016  1965             1   
1965 2016-06-16     11620     6      6  2016  1966             1   
1966 2016-06-17     11620     7      6  2016  1967             1   
1967 2016-06-18     11621     1      6  2016  1968             1   
1968 2016-06-19     11621     2      6  2016  1969            18   

      event_name_2  snap_CA  snap_TX  snap_WI  
0                1        0        0        0  
1                1        0        0        0  
2                1        0        0   

In [ ]:
duplicate_id_list1 = duplicate_id_list[:100]
# print(duplicate_id_list1)
cnt = 0
for instance in duplicate_id_list1:
  cnt += 1
  if cnt % 100 == 0:
    break
  filtered_df = grid_df[grid_df.id == instance]

  sales_ts = pd.DataFrame({
     'timestamps': filtered_df['d'].values,
     'demand': filtered_df['demand'].values
  })
  sales_ts['d'] = sales_ts['timestamps'] # keep sales_ts['d'] as integer

  # Merge với calendar_df theo cột 'd'
  sales_ts = sales_ts.merge(cal[['d', 'date']], on='d', how='left')

  sales_ts['timestamps'] = sales_ts['date']
  sales_ts = sales_ts.drop(columns=['date'])

  sales_ts['EMA'] = sales_ts['demand'].ewm(span=2, adjust=False).mean()
  sales_ts['EMA'] = sales_ts['EMA'].apply(math.ceil)

  from statsmodels.tsa.seasonal import STL
  from scipy import stats
  import numpy as np

  # Phân tích chuỗi theo mùa vụ
  stl = STL(sales_ts['demand'], period=30)  # nếu data theo ngày, 7 là 1 tuần
  res = stl.fit()

  residual = res.resid

  # Áp dụng z-score để tìm outlier trong residual
  z = np.abs(stats.zscore(residual))
  threshold = 2.8  # ngưỡng z-score
  outliers = sales_ts[z > threshold]
  # print(outliers.head())

  for idx, row in outliers.iterrows():
       grid_df.loc[(grid_df.id == instance) & (grid_df['d'] == row['d']), 'demand'] = row['EMA']

grid_df.to_parquet('../data/train_val1.parquet')

['HOBBIES_1_001_CA_1' 'HOBBIES_1_002_CA_1' 'HOBBIES_1_003_CA_1'
 'HOBBIES_1_004_CA_1' 'HOBBIES_1_005_CA_1' 'HOBBIES_1_006_CA_1'
 'HOBBIES_1_007_CA_1' 'HOBBIES_1_008_CA_1' 'HOBBIES_1_009_CA_1'
 'HOBBIES_1_010_CA_1' 'HOBBIES_1_011_CA_1' 'HOBBIES_1_012_CA_1'
 'HOBBIES_1_013_CA_1' 'HOBBIES_1_014_CA_1' 'HOBBIES_1_015_CA_1'
 'HOBBIES_1_016_CA_1' 'HOBBIES_1_017_CA_1' 'HOBBIES_1_018_CA_1'
 'HOBBIES_1_019_CA_1' 'HOBBIES_1_020_CA_1' 'HOBBIES_1_021_CA_1'
 'HOBBIES_1_022_CA_1' 'HOBBIES_1_023_CA_1' 'HOBBIES_1_024_CA_1'
 'HOBBIES_1_025_CA_1' 'HOBBIES_1_026_CA_1' 'HOBBIES_1_027_CA_1'
 'HOBBIES_1_028_CA_1' 'HOBBIES_1_029_CA_1' 'HOBBIES_1_030_CA_1'
 'HOBBIES_1_031_CA_1' 'HOBBIES_1_032_CA_1' 'HOBBIES_1_033_CA_1'
 'HOBBIES_1_034_CA_1' 'HOBBIES_1_035_CA_1' 'HOBBIES_1_036_CA_1'
 'HOBBIES_1_037_CA_1' 'HOBBIES_1_038_CA_1' 'HOBBIES_1_039_CA_1'
 'HOBBIES_1_040_CA_1' 'HOBBIES_1_041_CA_1' 'HOBBIES_1_042_CA_1'
 'HOBBIES_1_043_CA_1' 'HOBBIES_1_044_CA_1' 'HOBBIES_1_045_CA_1'
 'HOBBIES_1_046_CA_1' 'HOBBIES_1_047_CA_